<img src="https://teaching.bowyer.ai/courses/sdsai/resources/0/img/IMPERIAL_logo_RGB_Blue_2024.svg" alt="Imperial Logo" width="500"/><br /><br />

Machine Learning Foundations and Data Preparation
==============
### SURG70098 - Surgical Data Science and AI
### Stuart Bowyer

## Intended Learning Outcomes
1.  Describe what is meant by machine learning and the fundamental aspects of machine learning methods 
1.  Perform exploratory data analyses (EDA) of given datasets to understand how they can be used for machine learning
1.  Prepare datasets for use in machine learning by cleaning, transforming, and integrating them


## Session Outline
1.  [Machine Learning Foundations](#machine-learning-foundations)
1.  [Exploratory Data Analysis](#exploratory-data-analysis)
1.  [Data Cleaning](#data-cleaning)
1.  [Data Transformation](#data-transformation)
1.  [Data Integration](#data-integration)
1.  [Wrap Up](#wrap-up)

# Connecting to MIMIC
*   We will connect to the MIMIC databases using the `pandas-gbq` library
*   You should have already got access to MIMIC and setup a Google Cloud project before the session
*   To make the examples in this session work:
    *   Go to you [BigQuery project](https://console.cloud.google.com/bigquery) to get your `project_id`
    *   Input your `project_id` below, in my case it is `mimic-project-439312`
    *   Run this code block and it will store the datasets in your environment for use later on

In [ ]:
%pip install pandas_gbq

import pandas as pd
import pandas_gbq

project_id = 'mimic-project-439314'  # <--- INPUT YOUR PROJECT ID HERE

df_vitalsign = pandas_gbq.read_gbq("""
  SELECT *
  FROM `physionet-data.mimiciv_derived.vitalsign`
  LIMIT 10000
""", project_id=project_id)
 
df_day1_vitalsign = pandas_gbq.read_gbq("""
  SELECT *
  FROM `physionet-data.mimiciv_derived.first_day_vitalsign`
  LEFT JOIN (
    SELECT
      subject_id,
      stay_id,
      gender,
      race,
      admission_age,
      dod IS NOT NULL AS mortality
    FROM
      `physionet-data.mimiciv_derived.icustay_detail`
  )
  USING(subject_id, stay_id)
  WHERE heart_rate_mean IS NOT NULL
  LIMIT 10000
""", project_id=project_id)

df_bloodgas = pandas_gbq.read_gbq("""
  SELECT *
  FROM `physionet-data.mimiciv_derived.bg`
  LIMIT 10000
""", project_id=project_id)


# Machine Learning Foundations
*   The next four lectures of this course focus on the application of machine learning techniques to data
*   The definition we will follow for this course is:

> *"[when] a computer observes some data, builds a model based on the data, and uses the model as both a hypothesis about the world and a piece of software that can solve problems"*
>
> Artificial Intelligence a Modern Approach, Fourth Edition. Russel and Norvig. Pearson Education Limited, 2022

![ML illustration](https://teaching.bowyer.ai/courses/sdsai/resources/3/img/Illustrations-SDSAI-03_ML-Foundations.svg)

## ML is creating computer algorithms that learn through experience

## What Machine Learning is Not
*   Definitions of both vary, but machine learning is a subset of Artificial Intelligence
*   This does not include AI approaches that do not learn from data, including:
    *   Search algorithms
    *   Rule-based agents
    *   Logical agents

## Modelling Data
*   The concept of building models of data has been around for centuries - this is not new to ML
*   A ML (or statistical) model is one which allows you to understand and predict how different variables relate to each other
*   A model is generally simpler than the physical process they represent


![example of linear regression](https://upload.wikimedia.org/wikipedia/commons/3/3a/Linear_regression.svg)

## Modelling Data 

For a physical process:

$$y = f(X,Z)$$

where:
*   $y$ is the outputs ('dependent variable') - e.g. height
*   $f()$ is the physical process function
*   $X$ are the observed inputs ('indepdent variables') - e.g. age, ethnicity, nutrition, activity, etc.
*   $Z$ are the unobserved inputs ('indepdent variables') - e.g. omics, maternal exposures, etc.

The general form for a model of this is:

$$y = \hat{f}(X) + \epsilon$$

where:
*   $\hat{f}()$ is the model approximation of the physical process function
*   $\epsilon$ is the model's error - i.e. the combined effect of modelling error and unobserved variables


## Why Use a Model

![Illustration of data modelling](https://teaching.bowyer.ai/courses/sdsai/resources/3/img/Illustrations-SDSAI-03_Data-Modelling.svg)

Models can ...
*   Improve performance in sparse areas of the data
*   Respond better to noisy or missing data
*   Illucidate underlying mechanics
*   Have advantageous mathematical properties
*   Generalise to new problems

## What Makes a Good Model
For the previous form of a ML model: $y = \hat{f}(X) + \epsilon$

We define a 'good' model is one which minimises the value of the error $\epsilon$

**So, which of the following is the 'best' model? Why?**

![Illustration of data modelling](https://teaching.bowyer.ai/courses/sdsai/resources/3/img/Illustrations-SDSAI-03_Model-Overfitting.svg)

### Overfitting
*   The key here is minimising error on **future/unseen** data
*   When a model tightly fits to the training data such that it is inaccurate on new data, we call it 'overfitting'
*   We will consider many ways to avoid and address overfitting

![Illustration of data modelling](https://teaching.bowyer.ai/courses/sdsai/resources/3/img/Illustrations-SDSAI-03_Model-Overfitting.svg)


## Nomenclature
*   Machine learning resources often use a mixture of names for model components, borrowed from different places
*   Strictly, these concepts are not identical; however, we will use the red format for this course:

| **Role in Model**     | **Machine Learning (ML)**                 | **Statistics**            | **Regression Analysis**    |
|------------------------|------------------------------------------|---------------------------|----------------------------|
| **Input Variables**    | Predictors                               | Explanatory Variables     | Regressors                 |
|                        | <span style="color:red">Features</span>  | Covariates                | Independent Variables      |
|                        | Attributes                               |                           |                            |
|                        | Inputs                                   |                           |                            |
| **Output Variable**    | Target                                   | Response Variable         | Dependent Variable         |
|                        | <span style="color:red">Labels</span>    | Outcome                   |                            |
|                        | Output                                   | Criterion                 |                            |


# Exploratory Data Analysis
Understanding what data you have and how it could be used

## What is Exploratory Data Analysis

Exploratory data analysis (EDA) is the process of performing basic analyses of data to:
*   Identify what they (in the case of pre-/externally-collected data)
*   Validate your assumptions about them (i.e. do the data represent what they appear)
*   Identify outliers or anomalies
*   Generate initial analysis/modelling hypotheses

**This is critical to any subsequent analysis and modelling**

## If you were given a data set, how would you explore it for these purposes?

## Overview of EDA Methods

The following are **examples** of possible methods you can use in your EDA of different types of data ...

| **Category**      | **Univariate Methods** | **Multivariate Methods** |
|-------------------|------------------------|--------------------------|
| **Statistical**   | - Summary statistics (mean, median, mode, stdev, IQR, etc.)<br> - Distributions (normality, skewness, kurtosis)<br> - Frequency tables/counts | - Correlation matrix<br> - Covariance matrix<br> - Indepdence statistics (e.g. Chi-Squared)<br> - Difference statistics (e.g. ANOVA) |
| **Graphical**     | - Box plots<br> - Histograms<br> - Density plots<br> - Violin plots | - Scatter plots<br> - Pair plots (e.g. scatterplot matrix)<br> - Heatmaps<br> - Comparative/grouped distribution plots (e.g. grouped box-plots) |

## Univariate EDA
*   The purpose of these methods is to identify typical values/ranges for the data
*   This allows you to identify the:
    *   'Central tendancy' and range matches physical expectation
    *   Outliers
    *   Resolution of measurements
    *   Missing/substituted data
    *   Appropriate/compatible downstream models

### Statistical Univariate EDA Examples
#### Continuous Data

In [ ]:
df_day1_vitalsign.heart_rate_mean.mean()
df_day1_vitalsign.heart_rate_mean.median()
df_day1_vitalsign.heart_rate_mean.mode()
df_day1_vitalsign.heart_rate_mean.std()
df_day1_vitalsign.heart_rate_mean.max()
df_day1_vitalsign.heart_rate_mean.min()
df_day1_vitalsign.heart_rate_mean.skew()
df_day1_vitalsign.heart_rate_mean.kurt()

#### Categorical Data

In [ ]:

# Frequency count
df_day1_vitalsign.gender.value_counts()

# Frequency table
df_day1_vitalsign.gender.value_counts(normalize=True)

### Graphical Univariate EDA Examples
#### Continuous Data

In [ ]:
import matplotlib.pyplot as plt
fig, axes = plt.subplots(ncols=3,figsize=(12,4))
df_day1_vitalsign.boxplot(column='heart_rate_mean', ax = axes[0])
df_day1_vitalsign.heart_rate_mean.plot.hist(ax = axes[1])
df_day1_vitalsign.heart_rate_mean.plot.kde(ax = axes[2])           # You will need 'scipy' package installed
plt.show()


#### Categorical Data
*   Plotting value counts for discrete data can demonstrate what the data contain

In [ ]:
df_day1_vitalsign.race.value_counts() \
    .plot.bar()
plt.show()

## Multivariate EDA
*   The purpose of these methods is to explore potential relationships between different observations
*   This allows you to identify:
    *   Correlations
    *   Covariance

### Statistical Multivariate EDA Examples
#### Continuous Data
*   Pearson correlation coefficient identifies the strength and direction of a correlation between variables
    *   Positive (negative) values imply a positive (negative) correlation
    *   Values close to 1 (0) imply a strong (weak) correlation
    *   However, this test makes assumptions about linearity and normality


In [ ]:
df_day1_vitalsign[['heart_rate_mean','dbp_mean','heart_rate_max']].corr()

#### Categorical Data
*   Crosstabulation calculates the frequencies of all possible pairs of categorical variables
    *   This can highlight commen co-occurances
*   Chi-square test of independence is a test for significance in observed frequencies of categorical variables
    *   This can quantify dependencies between categorical variables

In [ ]:
from scipy.stats import chi2_contingency

# Crosstabulation
df_crosstab = pd.crosstab(df_day1_vitalsign['gender'], df_day1_vitalsign['mortality'])

# Chi-squared test
chi2, p, dof, _ = chi2_contingency(df_crosstab)

print(df_crosstab)
print(f"Chi-square statistic: {chi2}")
print(f"P-value: {p}")
print(f"Degrees of freedom: {dof}")

What does this result imply?

#### Mixed Continuous-Categorical Data
*   A range of statistical tests can identify significant differences in continuous data grouped on categorical data
*   For example, does gender influence day 1 SpO2
    *   The t-test is the standard benchmark; however, it has requirements for normality
    *   Mann-Whitney U Test is a non-parametric alternative

In [ ]:
from scipy.stats import mannwhitneyu
male_spo2 = df_day1_vitalsign.spo2_mean[df_day1_vitalsign.gender == 'M'].dropna()
female_spo2 = df_day1_vitalsign.spo2_mean[df_day1_vitalsign.gender == 'F'].dropna()

u_stat, p_value = mannwhitneyu(male_spo2, female_spo2)

print(f"Mann-Whitney U statistic: {u_stat}")
print(f"P-value: {p_value}")

### Graphical Multivariate EDA Examples
#### Continuous Data
*   For two-dimensional continuous data, the scatterplot is a simple way to initially understand possible interactions
*   For dense data, scatter plots can become difficult to interpret, heatmap plots are helpful here

In [ ]:
fig, axes = plt.subplots(ncols=2,figsize=(12,4))
df_day1_vitalsign.plot.scatter('sbp_mean', 'dbp_mean', ax=axes[0])
df_day1_vitalsign.plot.hexbin('sbp_mean', 'dbp_mean', gridsize=30, ax=axes[1])
plt.show()


*   For exploring many continuous variables together, you can use a pair plot (or scatterplot matrix) `pd.plotting.scatter_matrix()`
    *   This shows all pair-wise comparisons in the dataset as well as individual histograms
    *   This is very helpful when exploring independence and correlation

In [ ]:
pd.plotting.scatter_matrix(df_day1_vitalsign[['heart_rate_mean', 'sbp_mean', 'dbp_mean']])
plt.show()

#### Mixed Continuous-Categorical Data
*   Many plot types can be stratified by a categorical variable to explore differences

In [ ]:
fig, axes = plt.subplots(ncols=3,figsize=(12,4))
df_day1_vitalsign.boxplot(column='heart_rate_mean', by='gender', ax = axes[0])

df_day1_vitalsign[df_day1_vitalsign.gender == 'M'].heart_rate_mean.hist(label='M', ax = axes[1])
df_day1_vitalsign[df_day1_vitalsign.gender == 'F'].heart_rate_mean.hist(label='F', ax = axes[1])
axes[1].legend()

df_day1_vitalsign[df_day1_vitalsign.gender == 'M'].heart_rate_mean.plot.kde(label='M', ax = axes[2])
df_day1_vitalsign[df_day1_vitalsign.gender == 'F'].heart_rate_mean.plot.kde(label='F', ax = axes[2])
axes[2].legend()

plt.show()

## ❓ Exercise 3.1
[Colab notebook](https://colab.research.google.com/github/stuartbowyer/sdsai-lecture-notes/blob/main/Examples_Exercises/Exercise03.ipynb#scrollTo=exercise_3_1)

# Data Cleaning
Data cleaning is concerned with taking messy real-world data and making is appropriate for use in a model

## What Makes Data 'Messy'
Based on what you know about programming with data, what issues might there with using the following data to build a model?

| PatientID | Name         | Age   | Weight       | Height      | BloodPressure | Temp     | Diagnosis       | Gender  | AppointmentDate  |
|-----------|--------------|-------|--------------|-------------|---------------|----------|-----------------|---------|------------------|
| 101       | John         | 45    | 180 lbs      | 6'2"        | 120/80        | 98.6 F   | Hypertension    | M       | 2024-09-15       |
| 102       | Jane         |       | 70kg         | 165 cm      | 130/85        | 36.5 C   | Diabetes        | Female  | 15th Sep 2024    |
| 103       | Mike         | 53    | 250 lbs      | 72 in       | 140/90        | 100 F    | Hypertension    | M       | 2024-09-14       |
| 104       | Anna         | 401   | 50.5 kgs     | 5'4"        |               | 37 C     | Asthma          | F       | 09/16/2024       |
| 105       | Emily        | 60    | 130          | 160cm       |               |          | None            | Female  | 2024-09-13       |
| 106       | Chris        | -12   | 80kg         |             | 120/75        | 99.1 F   | Asthma          | Male    | 2024-09-12       |
| 107       | Sarah        | 40    | 165.3        | 5.5 ft      | 115/75        |          | Hypertension    | F       | September 14, 24 |
| 108       | Bob          | 67    | 200 lbs      |             |               |          | Heart Disease   | Male    | 2024/09/15       |
| 109       | Laura        | 32    | 140 lbs      | 1.70m       | 120/80        | 99F      | Hypertension    | Female  | 2024-09-14       |
| 110       | David        | 75    | 90kg         | 180 cm      | 110/70        | 37.5C    | None            | M       | 9/15/24          |


### Some Answers...
*   Units are inconsistent within the columns
*   Many missing values
*   Date formatting is inconsistent
*   Outliers (obvious in age)
*   Categorisations are inconsistent

| PatientID | Name         | Age   | Weight       | Height      | BloodPressure | Temp     | Diagnosis       | Gender  | AppointmentDate  |
|-----------|--------------|-------|--------------|-------------|---------------|----------|-----------------|---------|------------------|
| 101       | John         | 45    | 180 lbs      | 6'2"        | 120/80        | 98.6 F   | Hypertension    | M       | 2024-09-15       |
| 102       | Jane         |       | 70kg         | 165 cm      | 130/85        | 36.5 C   | Diabetes        | Female  | 15th Sep 2024    |
| 103       | Mike         | 53    | 250 lbs      | 72 in       | 140/90        | 100 F    | Hypertension    | M       | 2024-09-14       |
| 104       | Anna         | 401   | 50.5 kgs     | 5'4"        |               | 37 C     | Asthma          | F       | 09/16/2024       |
| 105       | Emily        | 60    | 130          | 160cm       |               |          | None            | Female  | 2024-09-13       |
| 106       | Chris        | -12   | 80kg         |             | 120/75        | 99.1 F   | Asthma          | Male    | 2024-09-12       |
| 107       | Sarah        | 40    | 165.3        | 5.5 ft      | 115/75        |          | Hypertension    | F       | September 14, 24 |
| 108       | Bob          | 67    | 200 lbs      |             |               |          | Heart Disease   | Male    | 2024/09/15       |
| 109       | Laura        | 32    | 140 lbs      | 1.70m       | 120/80        | 99F      | Hypertension    | Female  | 2024-09-14       |
| 110       | David        | 75    | 90kg         | 180 cm      | 110/70        | 37.5C    | None            | M       | 9/15/24          |

**What effect do you think these 'issues' will have on a model built from this data?**

## "Garbage In Garbage Out"

<table style='table-layout: fixed; width: 100%; margin-top: 0;'>
  <tbody>
    <tr>
      <td>
        <img src="https://teaching.bowyer.ai/courses/sdsai/resources/3/img/Garbage_in_garbage_out_gemini.png">
      </td>
      <td>
        <ul>
          <li>It will probably be clear that any model you build based on poor quality, messy data will produce poor quality results</li>
          <li>'Data Cleaning' is the process of improving your input data by correcting or removing errors and inconsistencies</li>
          <li>The following are some examples of key data cleaning methods; however, there is no limit to the type of data issues you can find in your data</li>
        </ul>
      </td>
    </tr>
  </tbody>
</table>

### Cleaning Medical Data
*   One of the biggest challenges in working with medical data is how 'messy' thay are
*   There are no definitive approaches to resolving any of these issues. You will need to use judgement about the:
    *   data type,
    *   data purpose, and
    *   <strong>importantly</strong> clinical context (you have an opportunity here)

## Missing Data
*   Missing data is one of the most common data issues you will find with medical data
*   Some common reasons for missing data are:
    *   Human factors (entry error, non-response, loss)
    *   Systemic factors (changing practice/guidelines, limited machine output)
    *   Availability (patient opt-out, research approvals, linkage, cost-of-acquisition)

❓ **Can you suggest how we might resolve missing data?**


| PatientID | Name         | Age   | Weight       | Height      | BloodPressure | Temp     | Diagnosis       | Gender  | AppointmentDate  |
|-----------|--------------|-------|--------------|-------------|---------------|----------|-----------------|---------|------------------|
| 102       | Jane         |       | 70kg         | 165 cm      | 130/85        | 36.5 C   | Diabetes        | Female  | 15th Sep 2024    |
| 104       | Anna         | 401   | 50.5 kgs     | 5'4"        |               | 37 C     | Asthma          | F       | 09/16/2024       |
| 105       | Emily        | 60    | 130          | 160cm       |               |          | None            | Female  | 2024-09-13       |
| 106       | Chris        | -12   | 80kg         |             | 120/75        | 99.1 F   | Asthma          | Male    | 2024-09-12       |
| 107       | Sarah        | 40    | 165.3        | 5.5 ft      | 115/75        |          | Hypertension    | F       | September 14, 24 |
| 108       | Bob          | 67    | 200 lbs      |             |               |          | Heart Disease   | Male    | 2024/09/15       |

#### Identifying Missing Data in pandas
*   The standard way to define missing data in pandas is with special symbols `NaN` or `None`
*   To find these, there is a pandas method `.isna()`
*   There is also the `.notna()` to find which elements are not missing

In [ ]:
df_messy = pd.read_csv("https://teaching.bowyer.ai/courses/sdsai/resources/3/data/messy_data.csv")
df_messy.isna()

#### Counting Missing Data in pandas
*   You can then find the number missing, per column, with `.sum()`

In [ ]:
df_messy.isna().sum()

### Dropping Missing Values

*   **What:** Exclude the missing data points from your data set
*   **How:** If the downstream model permits, you can drop an individual value, otherwise either drop the whole row or column
*   **Advantages:** This ensures your analysis/model/etc is based on the most accurate/representative data
*   **Disadvantages:** You often need to remove either the whole row or column, which wastes potentially useful data elements

#### Dropping Missing Data in pandas
*   To drop a row with missing data, pandas has the `.dropna()` method
*   Here you can specify which columns to look for missing data

In [ ]:
df_messy.dropna(subset='BloodPressure')

*   To drop a column with missing data, you can simply specify the column you wish to drop by name

In [ ]:
df_messy.drop('BloodPressure', axis=1)

### Imputation (Replacing Missing Values)
*   **What:** Imputation is the process of replacing a missing value with a substitute
*   **How:** A range of functions can be used to generate a substitute, from averages to predictive
*   **Advantages:** This maximises the available dataset
*   **Disadvantages:** Value substitution introduces inaccuracy and uncertainty into the data

#### Imputation Methods
##### Constant Value
*   Substitute missing values with a fixed constant (based on some context/knowledge)
*   ```df['A'].fillna(0)```

#### Average Value
*   Substutute missing values with the mean value of the non-missing values
*   ```df['B'].fillna(df['B'].mean())```
*   Could also be the `median()` or `mode()`

#### Random (Hot-deck)
*   Substitute missing values with a random selection from the available values
*   `rnd = pd.Series(np.random.choice(df['C'].dropna(), size=len(df['C'])))`
*   `df['C'].fillna(rnd)`

#### Regression
*   Substitute missing values with a predicted value based on a regression model of other (non-missing) variables

#### K-Nearest Neighbours
*   Substitute missing values with a predicted value based on a similarity model to other observations

#### Which Method to Use
Your choice of which imputation method to use will be based on:
*   **Domain knowledge** - what do the data represent, what assumptions are valid
*   **Data type** - numerical or categorical
*   **Amount of data** - 'small' data will probably limit accuracy of complex approaches
*   **Amount missing** - simple subsututions for a high proportion of missing data will introduce bias
*   **Downstream model requirements** - are models liable to overfitting

## ❓ Quick Exercise 3.2

**5 minutes**

[Colab notebook](https://colab.research.google.com/github/stuartbowyer/sdsai-lecture-notes/blob/main/Examples_Exercises/Exercise03.ipynb#scrollTo=exercise_3_2)

## Miss-Typed Data
*   It is very common for medical data to have been imported as a text string; however, data can often have their type changed
*   As a general rule, you want your types to be as simple as possible while maintaining the underlying meaning, i.e.:
    *   free text (`str`) → `float` → categorical (`str`) → `int` → `bool`
*   You should be careful of using integers to represent categorical data though, as it can imply ordinality that does not exist

### Type Conversions in pandas
*   We covered pandas type conversions extensively in lecture 2
*   Types can be easily modified in pandas with the `.astype()` or `pd.to_numeric()` methods
*   The `str` methods also allow for conversion of free text inputs to more constrained types

In [ ]:
df_bp = df_messy[['BloodPressure']].dropna()
df_bp[['Systolic','Diastolic']] = df_bp['BloodPressure'] \
    .str.split('/', expand=True) \
    .astype(int)

df_bp


## Inconsistent Data
*   Inconsistent data are those where different observations in a given column do not represent the same thing
*   A simple example would be measurements with different units (weight in kg or lbs) or taken in different ways (temperature taken internally or peripherally)
*   Data should be consistently represented to be useful for modelling and machine learning
*   Inconsistent data can be either:
    *   Converted to a common consistent format (e.g. converting temperatures to common units `degrees C`)
    *   Split into distinct observations (e.g. splitting to `internal_temp` and `peripheral_temp` observations)

## ❓ Quick Exercise 3.3

**5 minutes**

[Colab notebook](https://colab.research.google.com/github/stuartbowyer/sdsai-lecture-notes/blob/main/Examples_Exercises/Exercise03.ipynb#scrollTo=exercise_3_3)

## Outliers / Invalid Values

* Real clinical data often contain outliers or invalid entries — values that don’t make physiological or logical sense.
* These can distort statistical summaries and mislead learning algorithms.
* We must decide whether to correct, remove, or retain them depending on context and domain knowledge.

See examples of detecting and handling outliers in clinical datasets in the self learning appendices.

# Data Transformation
So far, we have considered how to get the given data into the most high-quality and representative. Data transformation is concerned with modifying that data to enrich it and make it more effective for subsequent modelling.

## Data Encoding
*   Consider that we want to build a (regression) predictive model based on 'age', 'weight', 'gender', 'ethnicity', and 'pain severity'.

*   The regression model is a mathematical equation and therefore (like many ML models) requires features be numerical, what could we do for the categorical data in 'gender', 'ethnicity', and 'pain severity'?

### Ordinal Encoding
*   Ordinal encoding assigns a numerical value to each category
*   If your numerical assignments are non-ordered, this is called 'label encoding'
*   For example, 'None', 'Mild', 'Moderate', 'Severe', and 'Very Severe' pain could be encoded as integers 0, 1, 2, 3, 4
*   Encoding data in this way embeds some assumptions:
    *   The data are ordinal (i.e. the model will infer a meaning in the order, e.g. 4 > 3 > 2 > 1 > 0)
    *   The data are linear (i.e. the model will infer a meaning in the linearity, e.g. 'moderate' is twice 'mild')

**Would this work for 'gender' or 'ethnicity'? Why?**

### One-hot Encoding
*   To avoid embedding some assumptions about ordinality/linearity of categorical data, you can use one-hot encoding
*   This creates a new feature for every possible value of a categorical feature, such that only one is 'true' for a given observation
*   The pandas function `pd.get_dummies()` can help with this

What does this encoding loose for the ethnicity data?

In [ ]:
pd.get_dummies(df_day1_vitalsign.race)

### Other Encoding
*   There are other encoding approaches, some that typically work better with higher cardinality data (i.e. those with more categories)
    *   Target encoding
    *   Frequency/count encoding
    *   Binary encoding
*   You can explore these in the reading list

## ❓ Quick Exercise 3.6

**5 minutes**

[Colab notebook](https://colab.research.google.com/github/stuartbowyer/sdsai-lecture-notes/blob/main/Examples_Exercises/Exercise03.ipynb#scrollTo=exercise_3_6)

## Feature Engineering
Feature engineering is the process of using some domain knowledge to create/extract new features from the existing data.

For example, if we have data on 'height' and 'weight', we can derive a 'BMI' feature

**As the ML model can just be given the base features, why do you think we might do this?**

### Why Engineer Features?
*   Embeds existing domain knowledge
*   Domain knowledge captures information not necessarily in a training set
*   Allows the use of simpler models
*   Improve model interpretability
*   Reduce model complexity/dimensionality
*   Therefore, reduce the risk of 'overfitting'

As clinical data scientists, you will have a significant opportunity for using domain knowledge to engineer new features that improve your models based on clinical understanding

### Engineering Derived Features
*   Deriving new features can be as simple as adding new columns to your data frame
*   For example, adding ...
    *   BMI
    *   An established risk score index
    *   Surgery duration
*   This code adds a new feature that specifies whether a patient has desaturated by 4% below their average baseline

In [ ]:
df_day1_vitalsign['desaturated'] = df_day1_vitalsign['spo2_min'] < df_day1_vitalsign['spo2_mean'] - 4.0

### Engineering Time Aggregated Feature
*   In some cases, you will want to aggregate measurements taken over a period of time to create a single feature
*   The pandas `groupby` can be used to calculate this
*   For example, the mean sodium blood gas measurement during an admission ... 

In [ ]:
df_bloodgas.groupby('hadm_id') \
    .sodium.mean() \
    .dropna()

### Engineering Binned Features
*   Sometimes you can simplify an existing feature by binning it into (fewer) categories
*   For numerical data, the pandas `cut()` function will help with this
*   For example, if you know that a specific model will be different for people under 30 and over 60, you can bin the age into three categories

In [ ]:
pd.cut(df_day1_vitalsign.admission_age, [0, 30, 60, 100])

### Feature Selection
*   Increasing features leads to something called the 'curse of dimensionality'
*   This is a result of increasing dimensionality leading to increasing training data sparseness leading to decreasing performance on new data
*   'Feature selection' is the process of picking individual features to use in a model
*   'Dimensionality reduction' is the process of creating new features that 'contain' the others
*   There are references on the curse of dimensionality and feature selection in the reading list for self study

## ❓ Quick Exercise 3.7

**10 minutes**

[Colab notebook](https://colab.research.google.com/github/stuartbowyer/sdsai-lecture-notes/blob/main/Examples_Exercises/Exercise03.ipynb#scrollTo=exercise_3_7)

## Feature Scaling

Machine learning algorithms are often sensitive to the scale of the data — especially those that rely on distance (e.g. k-NN, SVM, regression).
To ensure features contribute fairly to model training, we rescale them to comparable ranges.

Common approaches:
* Normalization: rescale values to 0–1 range
* Standardization: center data around 0 with unit variance

Explore examples of normalization and standardization in the self study appendices.

# Data Integration
It is very rare that all of the data you need for a good model will be stored in one place, data integration allows you to bring these separate data together

## Combining Data Sets
*   Most models will want a single data frame input containing all the features you want to model
*   In previous lectures we have seen how to combine data frames using `pandas.merge` and `pandas.concat`
*   If your data are from different sources, it is not uncommon to need to prepare them separately to get them into a common format that can be modelled
*   In general, you aim it to create a single feature table, as shown below ...

![Illustration of target structure for a data set](https://teaching.bowyer.ai/courses/sdsai/resources/3/img/Illustrations-SDSAI-03_Feature-Table.svg)

## Aligning Data Sets
*   There are many challenges you an occur when combining data sets
*   Aligning the data in time is a very common challenge
*   For example, some data will be recorded hourly, some daily

**If you were wanted to build a risk score model that used heart rate (recorded hourly) and lab test values (recorded ad-hoc) what do you think you would do?**

## ❓ Exercise 3.8

[Colab notebook](https://colab.research.google.com/github/stuartbowyer/sdsai-lecture-notes/blob/main/Examples_Exercises/Exercise03.ipynb#scrollTo=exercise_3_8)

# Wrap Up
*   This lecture has covered a range of steps used in the preparation of data for modelling and machine learning
*   There are no definitive rules for what a data preparation pipeline should look like
*   You will often need to revisit the data preparation once you start modelling as requirements and issues become clear

## Before Next Session

*   Revisit and complete all tutorial exercises. We will review these on Monday.

### New Material
*   Read reference on the 'curse of dimensionality' and feature selection methods

### Consolidation Reading
*   Read chapters 12, 13, and 15 of the 'Secondary Analysis of Electronic Health Records' book (on the reading list)
*   Read pages 51 to 88 of the 'Hands-on Machine Learning with Scikit-learn, Keras and TensorFlow' book (on the reading list)

# Self Study Appendices

## Outliers / Invalid Values
*   Outliers and invalid values (i.e. impossible/non-physiological values) can be introduced to data by human, acquisition, and transfer errors, as well as exceptional inputs
*   You can attempt to define outliers statistically; however, you should always start by considering the context
    *   i.e. can the value be negative, or greater than 100%, etc.

### Z-score Based Outlier Removal
*   If your data are normally (Gaussian) distributed z-scores can be used to identify outliers
*   The z-score is a number that represents how many 'standard deviations' a value is from the mean
*   Therefore, by setting a limit on the permissible z-score, values 'too far' from average can be removed
*   The z-score for a value can be calculated as follows:
$$Z = \frac{X - \mu}{\sigma}$$

where: $X$ is the value, $\mu$ is the mean of the column, $\sigma$ is the standard deviation of the column

<img src="https://teaching.bowyer.ai/courses/sdsai/resources/3/img/Illustrations-SDSAI-03_z-scores.svg" alt="z-score illustration" width="700"/>

#### Z-score Based Outlier Removal Example
*   The `temperature` values in the `vitalsign` MIMIC-IV dataset are approximately normal
*   With the following, we can eliminate values that are more than 3 standard deviations from the mean

In [ ]:
# Cast to numeric values
df_outlier = pd.DataFrame()
df_outlier['temp'] = pd.to_numeric(df_vitalsign['temperature'])

# Store original temp for comparison at the end
df_outlier['old_temp'] = df_outlier['temp']

# Compute z-scores using previous equation
df_outlier['zscore'] = (df_outlier['temp'] - df_outlier['temp'].mean()) / df_outlier['temp'].std()

# Replace extreme values with NaN by using the mask function
df_outlier['temp'] = df_outlier['temp'].mask(df_outlier['zscore'].abs() > 3.0)

# Check the result
df_outlier[['old_temp','temp']].describe()


### Interquartile Range (IQR) Based Outlier Removal
*   When your data are not normally distributed, the IQR can be used as it does not assume a specific distribution
*   The IQR is the range between the 25th and 75th percentiles in the data
*   Similarly to the z-scores, you can identify outliers by how many times the IQR they are from the first and third quartiles

<img src="https://teaching.bowyer.ai/courses/sdsai/resources/3/img/Illustrations-SDSAI-03_IQR.svg" alt="IQR illustration" width="700"/>


### Interquartile Range (IQR) Based Outlier Removal Example
*   The `glucose` values in the `vitalsign` MIMIC-IV dataset are non-normal
*   With the following, we can eliminate values that are more than 1.5 IQR from IQR

In [ ]:
# Cast to numeric values
df_outlier = pd.DataFrame()
df_outlier['glucose'] = pd.to_numeric(df_vitalsign['glucose'])

# Store original temp for comparison at the end
df_outlier['old_glucose'] = df_outlier['glucose']

# Compute IQR
q1 = df_outlier['glucose'].quantile(0.25)
q3 = df_outlier['glucose'].quantile(0.75)
iqr = q3 - q1

# Compute limits
lower_limit = q1 - 1.5 * iqr
upper_limit = q3 + 1.5 * iqr

# Replace extreme values with NaN by using the mask function
df_outlier['glucose'] = df_outlier['glucose'].mask(
    (df_outlier['glucose'] < lower_limit) | (df_outlier['glucose'] > upper_limit))

# Check the result
df_outlier[['old_glucose','glucose']].describe()

## ❓ Exercise 3.4

[Colab notebook](https://colab.research.google.com/github/stuartbowyer/sdsai-lecture-notes/blob/main/Examples_Exercises/Exercise03.ipynb#scrollTo=exercise_3_4)

## Feature Scaling
*   Machine learning methods do not typically perform well when the input feature data are of very difference scales
*   For example, consider a model based on white blood count (approx. 6000 cells/microliter) and insulin (10 mIU/mL), many models will be biased towards the higher value white blood count at the expense of the insulin
*   The same problem exists when scales are very 'lopsided'


### Data Normalisation
*   This is the process of transforming your numerical data into the range 0 to 1 (or sometimes -1 to 1)

$$ x_{normalised} = \frac{x - x_{min}}{x_{max} - x_{min}}$$

*   This avoids models being biased towards one feature over another
*   However, data with long tails will get very 'compressed'
*   This is also referred to as 'min-max scaling'

In [ ]:
rr = pd.DataFrame()
rr['raw'] = df_day1_vitalsign['resp_rate_mean']

# Normalise the 'raw' value
rr['norm'] = (rr.raw - rr.raw.min()) / \
    (rr.raw.max() - rr.raw.min())

# Plot
rr.plot.kde()
plt.show()

### Data Standardisation
*   For data with long tails (i.e. distant extremes) normalisation can cause issues with loss of resolution
*   Standardisation attempts to address this by scaling the values based on the normal distribution
*   The data are transformed to have zero mean, and unit (1) standard deviation

$$ x_{standardised} = \frac{x - \mu_{x}}{\sigma_{x}}$$

*   This is equivalent to computing z-scores

In [ ]:
rr = pd.DataFrame()
rr['raw'] = df_day1_vitalsign['resp_rate_mean']

# Standardise the 'raw' value
rr['norm'] = (rr.raw - rr.raw.mean()) /  rr.raw.std()

# Plot
rr.plot.kde()
plt.show()

### sklearn Functions
*   We will shortly start using the machine learning package `sklearn`, this has the following functions to perform normalisation and standardisation

In [ ]:
%pip install scikit-learn

In [ ]:
from sklearn.preprocessing import MinMaxScaler
from sklearn.preprocessing import StandardScaler

rr = pd.DataFrame()
rr['raw'] = df_day1_vitalsign['resp_rate_mean']

# Normalisation
rr['norm'] = MinMaxScaler() \
    .fit_transform(rr)

# Standardisation
rr['stan'] = StandardScaler() \
    .fit_transform(rr[['raw']])

# Plot
rr.plot.kde()
plt.show()


### Distribution Transformation
*   In addition to the previous data scaling issues, skewed data can cause challenges by biasing a model towards one side
*   Transformations can be applied to data to make them more even
*   The following example uses a log transform, but many more are possible

In [ ]:
import numpy as np

wbc = pd.DataFrame()
wbc['raw'] = df_day1_vitalsign.glucose_min

# Log-transform the data
wbc['log'] = np.log(wbc['raw'])

# Plot
fig, axes = plt.subplots(2, 1, figsize=(6, 6))
wbc['raw'].plot.kde(ax=axes[0], title="Raw")
wbc['log'].plot.kde(ax=axes[1], title="Log")
axes[0].set_xlim((0,400))
plt.show()

## ❓ Quick Exercise 3.5

**5 minutes**

[Colab notebook](https://colab.research.google.com/github/stuartbowyer/sdsai-lecture-notes/blob/main/Examples_Exercises/Exercise03.ipynb#scrollTo=exercise_3_5)